# ContractRisk M3 — evaluación FINAL sobre el gold (cuatro filas, ablaciones, juez y RAGAS)

**Sesión nueva de Colab con GPU T4** (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*). No requiere claves ni APIs pagas.

Qué hace, en orden: clona el repositorio en la rama indicada, instala versiones fijadas, corre las pruebas, reconstruye el índice Chroma desde cero, y ejecuta las 12 preguntas gold congeladas (`m3-qa-gold-1.0`) en:

1. línea base sin RAG (mismo generador Qwen2.5-3B, adaptación de la fila M2),
2. RAG ingenuo (denso top-3, S07),
3. RAG avanzado (híbrido BM25+denso con RRF y filtro de ID + reranking cross-encoder),
4. sistema final (avanzado + herramienta `diferencia_valores_contratos`, agente de máximo 3 pasos),
5. dos ablaciones (solo híbrido; denso + reranking).

Cada respuesta pasa por el harness `m3-harness-1.0` (exactitud, juez, dominio, latencia) y luego se calculan las cuatro métricas RAGAS (implementación del laboratorio S10, juez Qwen2.5-3B local) sobre ingenuo, avanzado y final. Duración esperada: 30–45 min. **Devuelva el ZIP de resultados y este notebook con salidas, incluso si algo falla.** No sustituya respuestas ni errores.

In [1]:
import os, subprocess, sys, tempfile, zipfile
from pathlib import Path
# Opción A (recomendada si la rama ya está publicada): clonar el repositorio.
# Opción B: USE_ZIP = True y subir contractrisk_m3_final_codigo.zip (generado con `git archive`).
USE_ZIP = True
REPO = 'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git'
BRANCH = 'Entrega/contractrisk-m3'
ROOT = Path('/content/contractrisk_m3_final')
os.chdir('/content')  # el directorio actual puede no existir si la celda se re-ejecutó tras borrar ROOT
if ROOT.exists():
    raise RuntimeError('Use una sesión nueva: la carpeta ya existe y la evidencia anterior no se sobrescribe')
if USE_ZIP:
    from google.colab import files
    uploaded = files.upload()  # se guarda en /content
    if len(uploaded) != 1:
        raise ValueError('Suba únicamente contractrisk_m3_final_codigo.zip')
    ROOT.mkdir(parents=True)
    with zipfile.ZipFile(Path('/content') / next(iter(uploaded))) as z:
        for member in z.infolist():
            if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Ruta no permitida en ZIP')
        z.extractall(ROOT)
    COMMIT = z.comment.decode() or 'zip-sin-commit'
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)], check=True)
    COMMIT = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
os.chdir(ROOT)
print('Código evaluado:', COMMIT)
assert Path('m3/run_final_m3.py').is_file(), 'El código no contiene la corrida final: use la rama o ZIP correctos'

Saving contractrisk_m3_final_codigo.zip to contractrisk_m3_final_codigo (1).zip
Código evaluado: 6026ac45494ba5a8342420c171720e414ebb6194


In [2]:
def run_visible(command, log_path):
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = proc.wait()
    if code:
        raise RuntimeError(f'Comando falló ({code}); consulte {log_path}')

run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], 'installation.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests.log')
run_visible([sys.executable, '-c', "import sys; sys.path.insert(0,'m3'); from sqlite_support import ensure_sqlite; print(ensure_sqlite()); import torch; assert torch.cuda.is_available(), 'Active GPU T4'; print(torch.cuda.get_device_name(0))"], 'environment.log')

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 148.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 32.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 165.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 133.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 161.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 173.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.0 M

In [3]:
# Corrida final. La carpeta de salida debe ser nueva.
run_visible([sys.executable, '-u', 'm3/run_final_m3.py', '--output', 'final_m3_resultados'], 'execution.log')

I0000 00:00:1793297209.389872    2255 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793297212.257678    2255 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Building dense Chroma index from scratch...
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
Loading fixed-revision reranker...

Fetching 7 files: 100%|██████████| 7/7 [00:08

In [4]:
# Tablas recalculadas desde el detalle por consulta.
from IPython.display import Markdown, display
display(Markdown(Path('final_m3_resultados/tablas_m3.md').read_text(encoding='utf-8')))

| Sistema | Dim. 1 exactitud (0-1) | Dim. 2 juez (1-5, válidos) | Dim. 3 dominio | Latencia mediana (s) |
|---|---:|---:|---:|---:|
| Línea base sin RAG (mismo generador; adaptación de M2) | 0.167 | 2.333 (12/12) | 2/12 (0.167) | 6.72 |
| RAG ingenuo (S07): denso top-3 | 0.250 | 3 (12/12) | 3/12 (0.250) | 8.00 |
| RAG avanzado: híbrido BM25+denso RRF con filtro de ID + reranking | 0.500 | 4 (12/12) | 4/12 (0.333) | 8.37 |
| Sistema final: avanzado + herramienta (agente acotado) | 0.611 | 4.167 (12/12) | 5/12 (0.417) | 8.86 |
| Ablación: híbrido RRF con filtro de ID, sin reranking | 0.500 | 3.917 (12/12) | 3/12 (0.250) | 7.61 |
| Ablación: denso + reranking, sin híbrido | 0.278 | 3.167 (12/12) | 3/12 (0.250) | 7.94 |

| Sistema | Faithfulness | Answer relevancy | Context precision | Context recall | Fuente gold recuperada |
|---|---:|---:|---:|---:|---:|
| RAG ingenuo (S07): denso top-3 | 0.642 (n=11) | 0.501 (n=10) | 0.033 (n=10) | 0.167 (n=10) | 0.200 |
| RAG avanzado: híbrido BM25+denso RRF con filtro de ID + reranking | 0.644 (n=12) | 0.317 (n=10) | 0.350 (n=10) | 0.592 (n=10) | 1.000 |
| Sistema final: avanzado + herramienta (agente acotado) | 0.708 (n=12) | 0.330 (n=10) | 0.383 (n=10) | 0.792 (n=10) | 1.000 |


In [5]:
# Trazas de la herramienta en el sistema final: decisiones reales del agente por pregunta.
import json
for line in Path('final_m3_resultados/harness_rows.jsonl').read_text(encoding='utf-8').splitlines():
    item = json.loads(line)
    row, rec = item['row'], item['record']
    if row['variant'] != 'sistema_final':
        continue
    decisions = [(e['step'], (e['decision'] or {}).get('action'), (e['tool_result'] or {}).get('ok'),
                  (e['tool_result'] or {}).get('operation') or (e['tool_result'] or {}).get('error'))
                 for e in rec.get('tool_trace', [])]
    out = rec.get('output') or {}
    print(row['eval_id'], row['kind'], '| llamadas:', rec.get('tool_calls'), '|', rec.get('termination'), '|', decisions)
    print('   respuesta:', out.get('respuesta'), '| abstencion:', out.get('abstencion'), '| dominio:', row['domain_pass'], row['domain_reasons'])

m3_001 factual | llamadas: 0 | model_answer | [(1, 'answer', None, None)]
   respuesta: Se contrata la construcción del cerramiento del aeropuerto Golfo de Morrosquillo de Tolú y lo hace la entidad AEROCIVIL. | abstencion: False | dominio: True []
m3_002 factual | llamadas: 0 | model_answer | [(1, 'answer', None, None)]
   respuesta: Se contrata la adquisición de elementos de protección personal para los servidores de la Regional Norte de Santander y lo hace la entidad AEROCIVIL. | abstencion: False | dominio: True []
m3_003 factual | llamadas: 0 | model_answer | [(1, 'answer', None, None)]
   respuesta: Se contrata la adquisición de equipo de investigación con destino a la Escuela de Infantería, y se contrata por parte de la Central Administrativa Regional Usaquén. | abstencion: False | dominio: False ['hechos_incompletos']
m3_004 factual | llamadas: 0 | model_answer | [(1, 'answer', None, None)]
   respuesta: Se contrata una consultoría y la entidad que lo contrata es el Instituto de

In [6]:
# Ejecute también tras un fallo para conservar diagnóstico.
import shutil
from google.colab import files
result_dir = ROOT / 'final_m3_resultados'
result_dir.mkdir(exist_ok=True)
(result_dir / 'commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, result_dir / log.name)
meta = result_dir / 'metadata.json'
print(meta.read_text(encoding='utf-8')[:3000] if meta.exists() else 'No se creó metadata: revisar logs.')
bundle = shutil.make_archive(str(ROOT / 'contractrisk_m3_final_resultados'), 'zip', result_dir)
files.download(bundle)

{
  "status": "completed",
  "purpose": "final_gold_evaluation_m3",
  "started_utc": "2026-10-29T18:06:35.825257+00:00",
  "python": "3.13.16",
  "models": {
    "embedding": [
      "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
      "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
    ],
    "generator": [
      "Qwen/Qwen2.5-3B-Instruct",
      "aa8e72537993ba99e69dfaafa59ed015b17504d1"
    ],
    "reranker": [
      "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
      "1427fd652930e4ba29e8149678df786c240d8825"
    ]
  },
  "config": {
    "config_version": "m3-final-1.0 (= integración v5 congelada)",
    "chunk_chars": 220,
    "overlap_chars": 30,
    "identity_in_embeddings": true,
    "top_k": 3,
    "candidate_count": 12,
    "rrf_constant": 60,
    "max_controller_steps": 3,
    "max_new_tokens": 512,
    "decoding": "greedy (do_sample=False)",
    "seed": 42
  },
  "labels": {
    "sin_rag": "Línea base sin RAG (mismo generador; adaptación de M2)",
    "rag_ingenuo"

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Devuelva `contractrisk_m3_final_resultados.zip` y este notebook descargado con sus salidas (*Archivo → Descargar → .ipynb*). Los resultados se versionan en `m3/results/final_m3/` y el reporte se escribe sobre ellos; no se repite la corrida para buscar mejores números.